# High level (Unitary) model training

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import torch
import torch.nn as nn
import seaborn as sns
import os
from dataProcesser import read_file_high_level, read_file_low_level , dicts_to_dataframe
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import clear_output
import time
import joblib

scaler = StandardScaler()

torch.manual_seed(42)
print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
# print(f"Current CUDA device: {torch.cuda.current_device()}")
# print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

## Importing Factory Reports

In [ ]:
df = []

senarios = list(range(1, 51))
demands = list(range(5, 1005, 5))
workers = ['00', '01']

for scenario in tqdm(senarios):
    for demand in demands:
        initial_conditions = f'initial_conditions/scenario_{scenario}/scenario_{scenario}_demand_{demand}.json'
        for worker in workers:
            simulation_result = f'simulation_report/worker_{worker}/scenario_{scenario}/scenario_{scenario}_demand_{demand}_report.json'
            df.append(read_file_high_level (initial_conditions, simulation_result))

df = dicts_to_dataframe(df)
df['output_per_time'] = df['total_output'] / df['total_time']
df.head()

## Plotting effect of treatment on the high level metrics

In [ ]:
# sns.set_palette("Set2")
from matplotlib import pyplot as plt
from matplotlib import ticker

In [ ]:

def plot_df(df):
    plt.style.use('ggplot')  # Using a standard Matplotlib style
    

    # ... (previous code remains the same) ...


    # font_size = 65
    # plt.rcParams.update({'font.size': font_size})
    # plt.rcParams.update({'legend.fontsize': font_size})
    # plt.rcParams.update({'axes.labelsize': font_size})
    # plt.rcParams.update({'axes.titlesize': font_size})
    # markersize = 30
    # linewidth = 7

    # Create a figure with 3 subplots
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(50, 20))

    # Plot 1: Total Output
    sns.kdeplot(data=df[df["treatment_id"] == 1], x="total_output", ax=ax1, label="Treatment 1", linewidth=linewidth)
    sns.kdeplot(data=df[df["treatment_id"] == 0], x="total_output", ax=ax1, label="Treatment 0", linewidth=linewidth)
    ax1.set_title("Total Output Distribution")
    ax1.set_xlabel("Total Output")
    # ax1.legend()

    # Plot 2: Total Time
    sns.kdeplot(data=df[df["treatment_id"] == 1], x="total_time", ax=ax2, label="Treatment 1", linewidth=linewidth)
    sns.kdeplot(data=df[df["treatment_id"] == 0], x="total_time", ax=ax2, label="Treatment 0", linewidth=linewidth)
    ax2.set_title("Total Time Distribution")
    ax2.set_xlabel("Total Time")
    # ax2.legend()

    # Plot 3: Output per Time
    sns.kdeplot(data=df[df["treatment_id"] == 1], x="output_per_time", ax=ax3, label="Treatment 1", linewidth=linewidth)
    sns.kdeplot(data=df[df["treatment_id"] == 0], x="output_per_time", ax=ax3, label="Treatment 0", linewidth=linewidth)
    ax3.set_title("Output per Time Distribution")
    ax3.set_xlabel("Output per Time")
    # have common single legend
    ax3.legend(loc='upper right', bbox_to_anchor=(1.5, 1.5), ncol=1)

    # Adjust the layout and display the plot
    plt.tight_layout()
    plt.savefig('output.pdf', bbox_inches='tight')
plot_df(df)

# Saving the reports as a csv file

In [ ]:
os.makedirs('csvs', exist_ok=True)
df.to_csv(f'csvs/factory_high_level.csv', index=False)

## Scaling of Variables

In [ ]:
processes = ['assembly', 'electronics', 'material-join', 'material-process']
inventory = ['raw_material', 'electronic_component', 'misc_component', 'fastener']
treatment = 'treatment_id'
# outcome = ['total_time', 'total_output']
outcome = ['total_output'] 
covariates = []
stations = []
# inputs.append(treatment)
covariates.extend(inventory)

for key in df.columns:
    if key.split('_')[0] in processes:
        stations.append(key)

covariates.extend(stations)

os.makedirs('Models', exist_ok=True)
os.makedirs('Models/Pytorch/', exist_ok=True)
os.makedirs('Models/Pytorch/HighLevel', exist_ok=True)

os.makedirs('Models/Scalers', exist_ok=True)
os.makedirs('Models/Scalers/HighLevel', exist_ok=True)

print(covariates)
with open("Models/Pytorch/HighLevel/covariates.json", "w") as f:
    json.dump(covariates, f)
with open("Models/Pytorch/HighLevel/outcome.json", "w") as f:
    json.dump(outcome, f)

In [ ]:
columns_to_scale = []
columns_to_scale.extend(inventory)
# columns_to_scale.extend(outcome)
df_scaled = df.copy(deep=True)
input_scaler = StandardScaler()
df_scaled[covariates] = input_scaler.fit_transform(df_scaled[covariates])
joblib.dump(input_scaler, 'Models/Scalers/HighLevel/HighLevelInputScaler.pkl')

output_scaler = MinMaxScaler()
df_scaled[outcome] = output_scaler.fit_transform(df_scaled[outcome])
df_apo = df.copy(deep=True)
df_apo_scaled = df.copy(deep=True)
df_apo_scaled[outcome] = output_scaler.transform(df_apo_scaled[outcome])
joblib.dump(output_scaler, 'Models/Scalers/HighLevel/HighLevelOutputScaler.pkl')

with open("Models/Scalers/HighLevel/columnstoscale-Input.json", "w") as f:
    json.dump(columns_to_scale, f)

display(df_scaled[columns_to_scale].describe())

In [ ]:
df_scaled["tree_depth"].value_counts()

## Initial EDA

In [ ]:
plot_df(df_scaled)

In [ ]:
dictDepth = {}
for d in range (df_scaled.tree_depth.min(), df_scaled.tree_depth.max()+1):
    dictDepth[f'{d}'] = len(df_scaled[df_scaled.tree_depth == d])

plt.figure(figsize=(20,6))
plt.bar(dictDepth.keys(), dictDepth.values())

## Test Train Split

In [ ]:
train, test  = train_test_split(df_scaled, test_size= 0.2, random_state= 420)
print(train.shape, ' ', test.shape)

## Model Training at different depths

In [ ]:
def observational_sampling(df_apo, biasing_covariate, bias_strength, plot_folder=None):
    df_apo.sort_values(by=["query_id", "treatment_id"], inplace=True)
    
    treatment_ids = list(df_apo["treatment_id"].unique())
    

    if biasing_covariate == "demand":
        # split query_id into scenario and demand
        df_apo["demand"] = df_apo["query_id"].apply(lambda x: int(x.split("_")[1]))
        cov = df_apo["demand"].values
    else:
        cov = df_apo[biasing_covariate].values
    # Apply overlap issues based on the covariate
    cov = cov[::2]  # Sample every second entry
    ecdf = ECDF(cov)
    cov_ecdf = ecdf(cov)
    cov_ecdf = cov_ecdf - np.mean(cov_ecdf)
    
    # Use the tree_depth per row to adjust bias strength if tree_depth exists, else default to 1
    # if "tree_depth" in df_apo.columns:
    #     tree_depths = df_apo["tree_depth"].values[::2]  # Get the corresponding tree depths for cov
    # else:
    #     tree_depths = np.ones(len(cov))  # If tree_depth doesn't exist, default to 1
    
    coefficients = np.repeat(bias_strength, len(cov))
    prob_values = 1 / (1 + np.exp(-coefficients * cov_ecdf))
    prob_values = np.clip(prob_values, 0.001, 0.999)
    assigned_treatment_ids = np.ra
    ndom.binomial(1, prob_values)
    assigned_treatment_ids = np.where(assigned_treatment_ids == 1, treatment_ids[0], treatment_ids[1])
    assigned_treatment_ids = np.repeat(assigned_treatment_ids, 2)
    df_apo["assigned_treatment_id"] = assigned_treatment_ids

    df_sampled = df_apo[df_apo["treatment_id"] == df_apo["assigned_treatment_id"]]
    df_cf_sampled = df_apo[~df_apo.index.isin(df_sampled.index)]
    
    # Drop the original treatment_id columns
    df_sampled.drop(columns=["treatment_id"], inplace=True)
    df_cf_sampled.drop(columns=["treatment_id"], inplace=True)
    
    # Rename the assigned treatment id to treatment_id for consistency
    df_sampled.rename(columns={"assigned_treatment_id": "treatment_id"}, inplace=True)
    df_cf_sampled.rename(columns={"assigned_treatment_id": "treatment_id"}, inplace=True)

    # Clean up the temporary column
    df_apo.drop(columns=["assigned_treatment_id"], inplace=True)
    
    # Create a dictionary of treatment assignments by query_id
    treatment_assignments = df_sampled[["query_id", "treatment_id"]].set_index("query_id").to_dict()["treatment_id"]
    
    return df_sampled, df_cf_sampled, treatment_assignments


In [ ]:
from statsmodels.distributions.empirical_distribution import ECDF
import warnings 
warnings.filterwarnings("ignore")
def train_test_split_by_depth(df, train_depths, test_depths):
    train_df = df[df['tree_depth'].isin(train_depths)]
    test_df = df[df['tree_depth'].isin(test_depths)]
    return train_df, test_df

def get_train_test(df, train_test_split_type, obs_biasing_type, knob_value):
    df_scaled = df.copy(deep=True)
    if obs_biasing_type == "random":
        df_sampled = df_scaled.sample(frac=1, random_state=42).groupby('query_id').head(1)
    elif obs_biasing_type == "tree_depth":
        df_sampled, df_cf_sampled, treatment_assignments = observational_sampling(df_scaled, "tree_depth", bias_strength=knob_value)
    elif obs_biasing_type == "demand":
        df_sampled, df_cf_sampled, treatment_assignments = observational_sampling(df_scaled, "demand", bias_strength=knob_value)

    if train_test_split_type == "random":
        train, test = train_test_split(df_sampled, test_size=0.2, random_state=42)
    elif train_test_split_type == "tree_depth":
        train_depths = np.arange(3, knob_value +1)
        test_depths = [8]
        print(f"Train depths: {train_depths}")
        print(f"Test depths: {test_depths}")
        train, test = train_test_split_by_depth(df_sampled, train_depths, test_depths)
    
    train_qids = train['query_id'].unique()
    test_qids = test['query_id'].unique()
    return train, test

In [ ]:
def get_train_test_from_file(df, knob_value, variable):
    print(knob_value, variable)
    hl_model_path = 'Models/Pytorch/HighLevel'
    with open(f'{hl_model_path}/{variable}/train_qids_{variable}_{knob_value}.json', 'r') as f:
        train_qids = json.load(f)
    with open(f'{hl_model_path}/{variable}/test_qids_{variable}_{knob_value}.json', 'r') as f:
        test_qids = json.load(f)
    with open(f'{hl_model_path}/{variable}/treatment_assignments_{variable}_{knob_value}.json', 'r') as f:
        treatment_assignments = json.load(f)
    print(len(train_qids), len(test_qids), len(treatment_assignments))
    print(test_qids)
    test = df[df['query_id'].isin(test_qids)]
    train = df[df['query_id'].isin(train_qids)]
    
    train["treatment_assigned"] = train["query_id"].map(treatment_assignments)
    train = train[train["treatment_assigned"] == train["treatment_id"]]
    train = train.drop(columns=["treatment_assigned"])
    
    
    return train, test

# Baseline models and model training

In [ ]:
class BaselineModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(BaselineModel, self).__init__()
        self.ff_network = nn.Sequential(
            nn.Linear(input_dim, int(input_dim*1.5)),
            nn.ReLU(),
            nn.Linear(int(input_dim*1.5), input_dim*2),
            nn.ReLU(),
            nn.Linear(input_dim*2, input_dim),
            nn.ReLU(),
            nn.Linear(input_dim, int(input_dim/2) if input_dim/2>output_dim else output_dim),
            nn.ReLU(),
            nn.Linear(int(input_dim/2) if input_dim/2>output_dim else output_dim, output_dim)
        )

    def forward(self, x):
        return self.ff_network(x)

from catenets.models.jax import TNet, SNet, DRNet, SNet1, SNet2
from econml.dml import NonParamDML
from sklearn.ensemble import RandomForestRegressor
from econml.metalearners import XLearner, TLearner, SLearner
def x_learner(df, covariates, treatment, outcome, X_test = None, output_scaler=None):
    # Split the data into features, treatment, and outcome
    X = df[covariates].values
    
    if len(covariates) == 1:
        X = X.reshape(-1, 1)
    
    T = df[treatment].values
    Y = df[outcome].values   
    est = XLearner(models=[RandomForestRegressor(), RandomForestRegressor()])
    est.fit(Y, T, X=X)

    # if X_test is None, then evaluate on the training data
    if X_test is None:
        X_test = X
    else:
    # otherwise evaluate on the test data
        X_test = X_test[covariates].values
        if len(covariates) == 1:
            X_test = X_test.reshape(-1, 1)
    causal_effect_estimates = est.effect(X_test)

    # scale back the output if output_scaler is provided
    if output_scaler is not None:
        causal_effect_estimates = output_scaler.inverse_transform(causal_effect_estimates.reshape(-1, 1)).flatten()
    
    return causal_effect_estimates

def s_learner(df, covariates, treatment, outcome, X_test = None, output_scaler=None):
    # Split the data into features, treatment, and outcome
    X = df[covariates].values
    
    if len(covariates) == 1:
        X = X.reshape(-1, 1)
    
    T = df[treatment].values
    Y = df[outcome].values   
    est = XLearner(models=RandomForestRegressor())
    est.fit(Y, T, X=X)

    # if X_test is None, then evaluate on the training data
    if X_test is None:
        X_test = X
    else:
    # otherwise evaluate on the test data
        X_test = X_test[covariates].values
        if len(covariates) == 1:
            X_test = X_test.reshape(-1, 1)
    causal_effect_estimates = est.effect(X_test)

    # scale back the output if output_scaler is provided
    if output_scaler is not None:
        causal_effect_estimates = output_scaler.inverse_transform(causal_effect_estimates.reshape(-1, 1)).flatten()
    
    return causal_effect_estimates

def t_learner(df, covariates, treatment, outcome, X_test = None, output_scaler=None):
    # Split the data into features, treatment, and outcome
    X = df[covariates].values
    
    if len(covariates) == 1:
        X = X.reshape(-1, 1)
    
    T = df[treatment].values
    Y = df[outcome].values   
    est = TLearner(models=[RandomForestRegressor(), RandomForestRegressor()])
    est.fit(Y, T, X=X)

    # if X_test is None, then evaluate on the training data
    if X_test is None:
        X_test = X
    else:
    # otherwise evaluate on the test data
        X_test = X_test[covariates].values
        if len(covariates) == 1:
            X_test = X_test.reshape(-1, 1)
    causal_effect_estimates = est.effect(X_test)

    # scale back the output if output_scaler is provided
    if output_scaler is not None:
        causal_effect_estimates = output_scaler.inverse_transform(causal_effect_estimates.reshape(-1, 1)).flatten()
    
    return causal_effect_estimates

def non_param_DML(df, covariates, treatment, outcome, X_test = None, output_scaler=None):
    # Split the data into features, treatment, and outcome
    X = df[covariates].values
    T = df[treatment].values
    Y = df[outcome].values

    est = NonParamDML(model_y=RandomForestRegressor(n_estimators=100, max_depth=10),
                  model_t=RandomForestRegressor(n_estimators=100, max_depth=10),
                  model_final=RandomForestRegressor(n_estimators=100, max_depth=10))
    est.fit(Y, T, X=X)

    # if X_test is None, then evaluate on the training data
    if X_test is None:
        X_test = X
    else:
    # otherwise evaluate on the test data
        X_test = X_test[covariates].values
    
    causal_effect_estimates = est.effect(X_test, T0=0, T1=1)
    # scale back the output if output_scaler is provided
    if output_scaler is not None:
        causal_effect_estimates = output_scaler.inverse_transform(causal_effect_estimates.reshape(-1, 1)).flatten()

    # this method only returns the effect estimates
    return causal_effect_estimates

def random_forest(df, covariates, treatment, outcome, X_test=None, output_scaler=None):
    # Split the data into features, treatment, and outcome
    X = df[covariates].values
    T = df[treatment].values
    Y = df[outcome].values
    print(X.shape, T.shape, Y.shape)

    # concatenate the features and treatment
    X_T = np.concatenate([X, T[:, None]], axis=1)
    # Fit the random forest models
    est = RandomForestRegressor(n_estimators=100, max_depth=10)
    est.fit(X_T, Y)

    if X_test is None:
        X_test = X
    else:
        X_test = X_test[covariates].values
    X_0 = np.concatenate([X_test, np.zeros((X_test.shape[0], 1))], axis=1)
    X_1 = np.concatenate([X_test, np.ones((X_test.shape[0], 1))], axis=1)
    # Predict the outcomes
    y1 = est.predict(X_1)
    y0 = est.predict(X_0)

    if output_scaler is not None:
        y1 = output_scaler.inverse_transform(y1.reshape(-1, 1)).flatten()
        y0 = output_scaler.inverse_transform(y0.reshape(-1, 1)).flatten()
    causal_effect_estimates = y1 - y0

    return causal_effect_estimates, y1, y0

def train_catenets(train_df, covariates, treatment, outcome, baseline="tnet"):
    X = train_df[covariates].values
    T = train_df[treatment].values
    Y = train_df[outcome].values
    if baseline == "tnet":
        est = TNet()
    elif baseline == "snet":
        est = SNet()
    elif baseline == "snet1":
        est = SNet1()
    elif baseline == "snet2":
        est = SNet2()
    elif baseline == "drnet":
        est = DRNet()
    est.fit(X,Y,T)

    return est

def predict_catenets(model, test_df, covariates, output_scaler=None, return_po=False):
    X_test = test_df[covariates].values
    causal_effect_estimates, y0, y1 = model.predict(X_test, return_po=True)
    if output_scaler is not None:
        y0 = output_scaler.inverse_transform(y0.reshape(-1, 1)).flatten()
        y1 = output_scaler.inverse_transform(y1.reshape(-1, 1)).flatten()
        causal_effect_estimates = y1 - y0
    return causal_effect_estimates, y1, y0
    

In [ ]:
def train_model(train_df, covariates, treatment, outcome, epochs, batch_size, val_df=None, plot=True):
    # use cuda if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training the model on {device}")

    input_dim = len(covariates)
    output_dim = len(outcome)

    # print(input_dim)
    model = BaselineModel(input_dim + 1, output_dim)
    # print(model)
    
    model.to(device)

    # shuffle the training data
    train_df = train_df.sample(frac=1).reset_index(drop=True)
    X = train_df[covariates].values
    T = train_df[treatment].values
    Y = train_df[outcome].values
    X_T = np.concatenate([X, T.reshape(-1, 1)], axis=1)
    print(X_T.shape)
    
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    X_T = torch.tensor(X_T, dtype=torch.float32).to(device)
    Y = torch.tensor(Y, dtype=torch.float32).reshape(-1, output_dim).to(device)
    
    # Prepare validation data if provided
    if val_df is not None:
        X_val = val_df[covariates].values
        T_val = val_df[treatment].values
        Y_val = val_df[outcome].values
        X_T_val = np.concatenate([X_val, T_val.reshape(-1, 1)], axis=1)
        X_T_val = torch.tensor(X_T_val, dtype=torch.float32)
        Y_val = torch.tensor(Y_val, dtype=torch.float32).reshape(-1, 1)
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for i in range(0, X_T.shape[0], batch_size):
            X_T_batch = X_T[i:i+batch_size]
            Y_batch = Y[i:i+batch_size]
            
            optimizer.zero_grad()
            output = model(X_T_batch)
            loss = loss_fn(output, Y_batch)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_train_loss = epoch_loss / (X_T.shape[0] // batch_size)
        train_losses.append(avg_train_loss)
        
        # Validation step
        if val_df is not None:
            model.eval()
            with torch.no_grad():
                val_output = model(X_T_val)
                val_loss = loss_fn(val_output, Y_val)
                val_losses.append(val_loss.item())
            
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss.item():.4f}")
        # else:
        #     print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}")
        
        # Plot losses every 10 epochs or at the end of training
        if ((epoch + 1) % 100 == 0 or epoch == epochs - 1) and plot == True:
            clear_output(wait=False)
            plt.clf()  # Clear the figure
            plt.figure(figsize=(10, 5))
            plt.plot(range(1, epoch + 2), train_losses, label='Training Loss')
            if val_df is not None:
                plt.plot(range(1, epoch + 2), val_losses, label='Validation Loss')
            plt.xlabel('Epochs')
            plt.ylabel('Loss')
            plt.title('Training and Validation Loss')
            plt.legend()
            plt.show()
    
    # return model, train_losses, val_losses
    return model

In [ ]:
device = 'cpu'
def get_pred (x, covariates, treatment, model):
    X = x[covariates].values
    T = x[treatment].values
    X_T = np.concatenate([X, T.reshape(-1, 1)], axis=1)
    X_T = torch.tensor(X_T, dtype=torch.float32).to(device)
    with torch.no_grad():
        predicted_effect = model(X_T).cpu().numpy()

    return predicted_effect

def get_pred_effect (x, covariates, treatment, model, output_scaler):
    predictions = {}
    X = x[covariates].values
    T = x[treatment].values
    X_1 = np.concatenate([X, np.ones(T.shape).reshape(-1, 1)], axis=1)
    X_0 = np.concatenate([X, np.zeros(T.shape).reshape(-1, 1)], axis=1)
    X_1 = torch.tensor(X_1, dtype=torch.float32).to(device)
    X_0 = torch.tensor(X_0, dtype=torch.float32).to(device)
    with torch.no_grad():
        p1 = model(X_1).cpu().numpy()
        p0 = model(X_0).cpu().numpy()
    if output_scaler is not None:
        p1 = output_scaler.inverse_transform(p1)
        p0 = output_scaler.inverse_transform(p0)
    predicted_effect = p1 - p0
    for i, qid in enumerate(x['query_id']):
        predictions[qid] = predicted_effect[i][0]

    return predictions

def get_pred_gt_effect(df_apo, outcome):
    gt_effects = {}
    df1 = df_apo[df_apo['treatment_id'] == 1]
    df0 = df_apo[df_apo['treatment_id'] == 0]
    df1 = df1[['query_id', outcome]]
    df0 = df0[['query_id', outcome]]
    # merge the two dataframes on query_id
    df_apo = pd.merge(df1, df0, on='query_id', suffixes=('_1', '_0'))
    effect = df_apo[outcome + '_1'] - df_apo[outcome + '_0']
    for i, qid in enumerate(df_apo['query_id']):
        gt_effects[qid] = effect[i]
    return gt_effects

def get_pred_gt (y):
    return y[outcome]

In [ ]:
def generate_train_test_splits(train_test_split_type, obs_biasing_type, variable, knob_values):
    model_path = f'Models/Pytorch/HighLevel/{variable}'
    if not os.path.exists(model_path):
        os.makedirs(model_path)
    for knob_value in knob_values:
        train, test = get_train_test(df_scaled, train_test_split_type, obs_biasing_type, knob_value)
        # save train qids and test qids
        train_qids = train['query_id'].unique()
        test_qids = test['query_id'].unique()
        
        # get assigned treatment ids and save them as well
        treatment_assignments = train[["query_id", "treatment_id"]].set_index("query_id").to_dict()["treatment_id"]
        with open(f'{model_path}/train_qids_{variable}_{knob_value}.json', 'w') as f:
            json.dump(train_qids.tolist(), f)
        with open(f'{model_path}/test_qids_{variable}_{knob_value}.json', 'w') as f:
            json.dump(test_qids.tolist(), f)
        with open(f'{model_path}/treatment_assignments_{variable}_{knob_value}.json', 'w') as f:
            json.dump(treatment_assignments, f)



# Generate train/test splits for various experiments 
train_test_split_type = "tree_depth"
knob_values = [3, 4, 5, 6, 7, 8]
variable = 'tree_depth'
obs_biasing_type = "random"
# generate split for CG experiment
generate_train_test_splits(train_test_split_type, obs_biasing_type, variable, knob_values )

# generate split for observational bias experiment
train_test_split_type = "random"
knob_values = list(range(0, 11))
variable = 'bias_strength'
obs_biasing_type = "tree_depth"

# Sample Size Experiment (Figure 2 (a))

In [ ]:
dictOfModels = {}
gt_effects = get_pred_gt_effect(df_apo, outcome[0])
gt_effects_scaled = get_pred_gt_effect(df_apo_scaled, outcome[0])
knob_values = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
for model_name in ['tnet', 'random_forest', 'non_param_DML', 'x_learner', 'neural_net']:
        train_orig, test = get_train_test_from_file(df_scaled, 0, 'bias_strength')
        for knob_value in knob_values:
            # have train_size as 0.1*train_size
            variable = 'sample_size'
            print(f'training for variable {variable} with model {model_name} and knob value {knob_value}')
            train = train_orig.sample(frac=knob_value, random_state=42)
            if model_name == 'tnet':
                model = train_catenets(train, covariates, treatment, outcome, baseline="tnet")
                estimates, y1, y0 = predict_catenets(model, test, covariates, output_scaler=output_scaler, return_po=True)
            elif model_name == 'random_forest':
                estimates, y1, y0 = random_forest(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
            elif model_name == 'non_param_DML':
                estimates = non_param_DML(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
            elif model_name == 'x_learner':
                estimates = x_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
            elif model_name == 's_learner':
                estimates = s_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
            elif model_name == 't_learner':
                estimates = t_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
            elif model_name == 'neural_net':
                # model = torch.load(f'Models/Pytorch/HighLevel/{variable}/model_{variable}-{knob_value}.pth', weights_only=False)
                model = train_model(train, covariates, treatment, outcome, epochs=100, batch_size=len(train), plot=True)
                estimates = get_pred_effect(test, covariates, treatment, model, output_scaler)

            if model_name in ['tnet', 'random_forest', 'neural_net']:
                gt_effects = get_pred_gt_effect(df_apo, outcome[0])
            else:
                gt_effects = get_pred_gt_effect(df_apo, outcome[0])

            if model_name != 'neural_net':
                estimates = {test['query_id'].iloc[i]: estimates[i] for i in range(len(estimates))}
            pred_df = pd.DataFrame(list(estimates.items()), columns=['query_id', 'predicted_effect'])
            actual_df = pd.DataFrame(list(gt_effects.items()), columns=['query_id', 'actual_effect'])
            df_effect = pd.merge(pred_df, actual_df, on='query_id')
            error = mean_squared_error(df_effect['actual_effect'], df_effect['predicted_effect'])
            r2 = r2_score(df_effect['actual_effect'], df_effect['predicted_effect'])
            dictOfModels[f'{model_name}_{variable}_{knob_value}'] = {'error': error, 'r2': r2}

# Compositional Generalization Experiment (Figure 2(b))

In [ ]:
error = {}
r2= {}
train_test_split_type = "tree_depth"
obs_biasing_type = "random"
if train_test_split_type == "tree_depth":
    knob_values = [3, 4, 5, 6, 7, 8]
    variable = 'tree_depth'
else:
    knob_values = list(range(0, 11))
    variable = 'bias_strength'
gt_effects = get_pred_gt_effect(df_apo, outcome[0])
gt_effects_scaled = get_pred_gt_effect(df_apo_scaled, outcome[0])

for model_name in ['tnet', 'random_forest', 'non_param_DML', 'x_learner', 'neural_net']:
   
    for knob_value in knob_values:
        # have train_size as 0.1*train_size
        train, test = get_train_test_from_file(df_scaled, knob_value, variable)
        if model_name == 'tnet':
            model = train_catenets(train, covariates, treatment, outcome, baseline="tnet")
            estimates, y1, y0 = predict_catenets(model, test, covariates, output_scaler=output_scaler, return_po=True)
        elif model_name == 'random_forest':
            estimates, y1, y0 = random_forest(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 'non_param_DML':
            estimates = non_param_DML(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 'x_learner':
            estimates = x_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 's_learner':
            estimates = s_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 't_learner':
            estimates = t_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 'neural_net':
            # model = torch.load(f'Models/Pytorch/HighLevel/{variable}/model_{variable}-{knob_value}.pth', weights_only=False)
            model = train_model(train, covariates, treatment, outcome, epochs=100, batch_size=len(train), plot=True)
            estimates = get_pred_effect(test, covariates, treatment, model, output_scaler)

        if model_name in ['tnet', 'random_forest', 'neural_net']:
            gt_effects = get_pred_gt_effect(df_apo, outcome[0])
        else:
            gt_effects = get_pred_gt_effect(df_apo, outcome[0])

        if model_name != 'neural_net':
            estimates = {test['query_id'].iloc[i]: estimates[i] for i in range(len(estimates))}
        pred_df = pd.DataFrame(list(estimates.items()), columns=['query_id', 'predicted_effect'])
        actual_df = pd.DataFrame(list(gt_effects.items()), columns=['query_id', 'actual_effect'])
        df_effect = pd.merge(pred_df, actual_df, on='query_id')
        error = mean_squared_error(df_effect['actual_effect'], df_effect['predicted_effect'])
        r2 = r2_score(df_effect['actual_effect'], df_effect['predicted_effect'])
        dictOfModels[f'{model_name}_{variable}_{knob_value}'] = {'error': error, 'r2': r2}

# Observational Bias Experiment (Figure 2(c))

In [ ]:
error = {}
r2= {}
train_test_split_type = "random"
obs_biasing_type = "tree_depth"
if train_test_split_type == "tree_depth":
    knob_values = [3, 4, 5, 6, 7, 8]
    variable = 'tree_depth'
else:
    knob_values = list(range(0, 11))
    variable = 'bias_strength'
gt_effects = get_pred_gt_effect(df_apo, outcome[0])
gt_effects_scaled = get_pred_gt_effect(df_apo_scaled, outcome[0])

for model_name in ['tnet', 'random_forest', 'non_param_DML', 'x_learner', 'neural_net']:
   
    for knob_value in knob_values:
        # have train_size as 0.1*train_size
        train, test = get_train_test_from_file(df_scaled, knob_value, variable)
        if model_name == 'tnet':
            model = train_catenets(train, covariates, treatment, outcome, baseline="tnet")
            estimates, y1, y0 = predict_catenets(model, test, covariates, output_scaler=output_scaler, return_po=True)
        elif model_name == 'random_forest':
            estimates, y1, y0 = random_forest(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 'non_param_DML':
            estimates = non_param_DML(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 'x_learner':
            estimates = x_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 's_learner':
            estimates = s_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 't_learner':
            estimates = t_learner(train, covariates, treatment, outcome[0], X_test=test, output_scaler=output_scaler)
        elif model_name == 'neural_net':
            # model = torch.load(f'Models/Pytorch/HighLevel/{variable}/model_{variable}-{knob_value}.pth', weights_only=False)
            model = train_model(train, covariates, treatment, outcome, epochs=100, batch_size=len(train), plot=True)
            estimates = get_pred_effect(test, covariates, treatment, model, output_scaler)

        if model_name in ['tnet', 'random_forest', 'neural_net']:
            gt_effects = get_pred_gt_effect(df_apo, outcome[0])
        else:
            gt_effects = get_pred_gt_effect(df_apo, outcome[0])

        if model_name != 'neural_net':
            estimates = {test['query_id'].iloc[i]: estimates[i] for i in range(len(estimates))}
        pred_df = pd.DataFrame(list(estimates.items()), columns=['query_id', 'predicted_effect'])
        actual_df = pd.DataFrame(list(gt_effects.items()), columns=['query_id', 'actual_effect'])
        df_effect = pd.merge(pred_df, actual_df, on='query_id')
        error = mean_squared_error(df_effect['actual_effect'], df_effect['predicted_effect'])
        r2 = r2_score(df_effect['actual_effect'], df_effect['predicted_effect'])
        dictOfModels[f'{model_name}_{variable}_{knob_value}'] = {'error': error, 'r2': r2}

In [ ]:
# save dictOfModels to a file
with open(f"{model_path}/high_level_results.json", "w") as f:
    json.dump(dictOfModels, f, indent = 2)